# V9 Checkpoint Real-Load Preflight T4x2

`NO_FAKE_RESULTS`  
`NO_REAL_EVIDENCE`  
`not paper evidence`  
`claim_allowed=false`

Preflight images are `run_log_only` and are never evidence.

Resume is intentionally unsupported. Reruns are non-destructive: choose a clean Kaggle session; existing output directories or ZIPs are refused.

In [ ]:
import hashlib, json, shutil, subprocess, sys, time, zipfile
from pathlib import Path
try:
    import torch
    print('gpu_count', torch.cuda.device_count())
    print('gpu_names', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    if torch.cuda.device_count() < 2:
        raise RuntimeError('PREFLIGHT_BLOCKED_CUDA')
except Exception as exc:
    Path('/kaggle/working/checkpoint_preflight_blocked_status.json').write_text(json.dumps({'status_code':'PREFLIGHT_BLOCKED_CUDA','error':str(exc),'claim_allowed':False}, indent=2))
    raise
print('python', sys.version)
print('disk', shutil.disk_usage('/kaggle/working'))

In [ ]:
!pip -q install 'diffusers==0.34.0' 'transformers==4.53.2' 'accelerate==1.8.1' 'safetensors==0.5.3' 'pillow==11.2.1'
!python -m pip freeze > /kaggle/working/preflight_dependency_freeze.txt

In [ ]:
from diffusers import DDPMPipeline
MODELS = [('google/ddpm-cifar10-32','google_ddpm','267b167dc01f0e4e61923ea244e8b988f84deb80'), ('FrankCCCCC/ddpm_ema_cifar10','frank_ddpm_ema','6aa387f240fbb00d0e003f93a3b994f56dd98dc2'), ('FrankCCCCC/cfm-cifar10-32','frank_cfm','b3f30358497e11ce5011c00614c9b0521262f51c')]
OUT = Path('/kaggle/working/checkpoint_preflight')
OUT.mkdir(parents=True, exist_ok=False)
results=[]
for model_index, (checkpoint, short_id, revision) in enumerate(MODELS):
    start=time.time()
    try:
        gpu = model_index % 2
        pipe = DDPMPipeline.from_pretrained(checkpoint, revision=revision).to(f'cuda:{gpu}')
        images = pipe(batch_size=2, num_inference_steps=4).images
        model_dir = OUT / short_id
        model_dir.mkdir(parents=True, exist_ok=True)
        for idx, image in enumerate(images):
            image.save(model_dir / f'preflight_{idx:02d}.png')
        results.append({'checkpoint_id':checkpoint,'checkpoint_revision':revision,'short_id':short_id,'gpu':gpu,'status_code':'PREFLIGHT_PASS','wall_time_seconds':time.time()-start,'evidence_status':'run_log_only','claim_allowed':False})
    except ImportError as exc:
        results.append({'checkpoint_id':checkpoint,'checkpoint_revision':revision,'short_id':short_id,'status_code':'PREFLIGHT_BLOCKED_DEPENDENCY','error':str(exc),'wall_time_seconds':time.time()-start,'claim_allowed':False})
    except RuntimeError as exc:
        status = 'PREFLIGHT_BLOCKED_CUDA' if 'cuda' in str(exc).lower() else 'PREFLIGHT_BLOCKED_MODEL_LOAD'
        results.append({'checkpoint_id':checkpoint,'checkpoint_revision':revision,'short_id':short_id,'status_code':status,'error':str(exc),'wall_time_seconds':time.time()-start,'claim_allowed':False})
    except Exception as exc:
        status = 'PREFLIGHT_BLOCKED_SCHEDULER' if 'scheduler' in str(exc).lower() else 'PREFLIGHT_BLOCKED_MODEL_LOAD'
        results.append({'checkpoint_id':checkpoint,'checkpoint_revision':revision,'short_id':short_id,'status_code':status,'error':str(exc),'wall_time_seconds':time.time()-start,'claim_allowed':False})
payload={'status_code':'PREFLIGHT_PASS' if all(r['status_code']=='PREFLIGHT_PASS' for r in results) else 'PREFLIGHT_BLOCKED_MODEL_LOAD','results':results,'evidence_status':'run_log_only','claim_allowed':False}
Path('/kaggle/working/checkpoint_preflight_status.json').write_text(json.dumps(payload, indent=2))
payload

In [ ]:
integrity=[]
for path in sorted(Path('/kaggle/working/checkpoint_preflight').rglob('*')):
    if path.is_file(): integrity.append({'path':str(path.relative_to('/kaggle/working')),'size':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
integrity.extend([{'path':'checkpoint_preflight_status.json','size':Path('/kaggle/working/checkpoint_preflight_status.json').stat().st_size,'sha256':hashlib.sha256(Path('/kaggle/working/checkpoint_preflight_status.json').read_bytes()).hexdigest()},{'path':'preflight_dependency_freeze.txt','size':Path('/kaggle/working/preflight_dependency_freeze.txt').stat().st_size,'sha256':hashlib.sha256(Path('/kaggle/working/preflight_dependency_freeze.txt').read_bytes()).hexdigest()}])
Path('/kaggle/working/checkpoint_preflight/output_zip_integrity_manifest.json').write_text(json.dumps({'files':integrity,'claim_allowed':False}, indent=2))
ZIP=Path('/kaggle/working/certgen_checkpoint_preflight_outputs.zip')
if ZIP.exists(): raise FileExistsError(f'refusing to overwrite {ZIP}')
with zipfile.ZipFile(ZIP, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(Path('/kaggle/working/checkpoint_preflight').rglob('*')):
        if path.is_file(): archive.write(path, path.relative_to('/kaggle/working'))
    archive.write('/kaggle/working/checkpoint_preflight_status.json', 'checkpoint_preflight_status.json')
    archive.write('/kaggle/working/preflight_dependency_freeze.txt', 'preflight_dependency_freeze.txt')
print('Copy back /kaggle/working/certgen_checkpoint_preflight_outputs.zip to data/kaggle_outputs/')
print('Then run commands/v9_cpu_execution/02_import_checkpoint_preflight_zip.sh')